# CIFAR-100 CNN with VVTKDataset + PyTorch DataLoader

1. Download CIFAR-100 from torchvision
2. Store train/test splits into VVTK sharded binary dataset (`./data`)
3. Train a simple CNN with Adam for 10 epochs using the custom dataset

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchvision import datasets
from vvtk_dataset import VVTKDataset

## Step 1 — Download CIFAR-100 and store into VVTK shards

In [ ]:
# Download CIFAR-100
train_cifar = datasets.CIFAR100(root='data/cifar100_raw', train=True,  download=True)
test_cifar  = datasets.CIFAR100(root='data/cifar100_raw', train=False, download=True)

print(f'Train samples: {len(train_cifar)}')
print(f'Test  samples: {len(test_cifar)}')
print(f'Image shape:   {np.array(train_cifar[0][0]).shape}')  # (32, 32, 3)

100%|██████████| 169M/169M [00:32<00:00, 5.16MB/s] 


Train samples: 50000
Test  samples: 10000
Image shape:   (32, 32, 3)


In [ ]:
def store_to_vvtk(cifar_dataset, path, num_shards=16):
    """Store a torchvision CIFAR dataset into VVTK binary shards.
    Each sample is a tuple of (image_uint8 [3,32,32], label_int64 [1]).
    """
    with VVTKDataset(path, mode='wb', num_shards=num_shards,
                     compression=['none', 'none']) as ds:
        for idx in range(len(cifar_dataset)):
            img_pil, label = cifar_dataset[idx]
            # HWC uint8 -> CHW uint8
            img = np.array(img_pil, dtype=np.uint8).transpose(2, 0, 1)
            lbl = np.array([label], dtype=np.int64)
            ds.add(idx, img, lbl)
    print(f'Saved {len(cifar_dataset)} samples to {path}')

store_to_vvtk(train_cifar, 'data/shards/train', num_shards=16)
store_to_vvtk(test_cifar,  'data/shards/test',  num_shards=4)

[VVTK] Saved dataset to ./data/train
Saved 50000 samples to ./data/train
[VVTK] Saved dataset to ./data/test
Saved 10000 samples to ./data/test


## Step 2 — Read back with VVTKDataset + standard PyTorch DataLoader

In [ ]:
class CIFAR100VVTK(torch.utils.data.Dataset):
    """Thin wrapper that adapts VVTKDataset to a standard torch Dataset."""
    def __init__(self, path):
        self.ds = VVTKDataset(path, mode='rb', compression=['none', 'none'])

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        img, lbl = self.ds[idx]           # uint8 [3,32,32], int64 [1]
        img = img.float().div(255.0)       # normalize to [0,1]
        lbl = lbl.squeeze().long()         # scalar label
        return img, lbl

train_ds = CIFAR100VVTK('data/shards/train')
test_ds  = CIFAR100VVTK('data/shards/test')

# Verify
img, lbl = train_ds[0]
print(f'Image: {img.shape} {img.dtype}  |  Label: {lbl.item()}')

Image: torch.Size([3, 32, 32]) torch.float32  |  Label: 19


In [5]:
BATCH_SIZE = 128

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
test_loader  = torch.utils.data.DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

## Step 3 — Define a simple CNN

In [6]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 32x32 -> 16x16
            nn.Dropout(0.2),

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 16x16 -> 8x8
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2),                                          # 8x8 -> 4x4
            nn.Dropout(0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)
print(f'Device: {device}  |  Params: {sum(p.numel() for p in model.parameters()):,}')

Device: cuda  |  Params: 838,148


## Step 4 — Train for 10 epochs

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

NUM_EPOCHS = 10

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += imgs.size(0)

    # --- Eval ---
    model.eval()
    test_correct, test_total = 0, 0
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = model(imgs).argmax(1)
            test_correct += (preds == labels).sum().item()
            test_total += imgs.size(0)

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss/train_total:.4f}  Acc: {100*train_correct/train_total:.1f}% | '
          f'Test Acc: {100*test_correct/test_total:.1f}%')

Epoch  1/10 | Train Loss: 4.1928  Acc: 5.2% | Test Acc: 12.7%
Epoch  2/10 | Train Loss: 3.7788  Acc: 10.4% | Test Acc: 16.8%
Epoch  3/10 | Train Loss: 3.5584  Acc: 13.4% | Test Acc: 21.5%
Epoch  4/10 | Train Loss: 3.3755  Acc: 16.4% | Test Acc: 24.5%
Epoch  5/10 | Train Loss: 3.2302  Acc: 18.5% | Test Acc: 27.0%
Epoch  6/10 | Train Loss: 3.1160  Acc: 20.6% | Test Acc: 27.9%
Epoch  7/10 | Train Loss: 3.0239  Acc: 22.0% | Test Acc: 30.4%
Epoch  8/10 | Train Loss: 2.9426  Acc: 23.8% | Test Acc: 34.6%
Epoch  9/10 | Train Loss: 2.8622  Acc: 25.0% | Test Acc: 34.9%
Epoch 10/10 | Train Loss: 2.7882  Acc: 26.6% | Test Acc: 35.5%
